# GLM-5.3-Flash 无审查版（Maya-S-v2 + 去审查 LoRA）· Strata-GLM

- 模型：[peasantsmith/GLM-5.3-Flash-Maya-GGUF](https://huggingface.co/peasantsmith/GLM-5.3-Flash-Maya-GGUF) 的 `Maya-S-v2-IQ2_XXS`（320B MoE / 18B 激活，约 96GB）
- 去审查：[GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF](https://huggingface.co/GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF) 的 `Ablitered2-LoRA-v2`（启动时加载，不改模型文件）
- 推理引擎：[Indras-Mirror/Strata-GLM](https://github.com/Indras-Mirror/Strata-GLM)（热门专家放显存，其余放内存池）

需要 **G4 运行时**（96GB 显存 + 176GB 内存）。依次运行：
1. 挂载 Drive，从 Hugging Face 下载模型和 LoRA（约 10 分钟）
2. 准备引擎：Drive 里有编译好的 `glm_generate` 就复用；没有就编译（约 25 分钟，编完存回 Drive）
3. 启动服务（端口 8001，512K 上下文；首次加载约 5 分钟）
4. 保活代理 8000 → 8001
5. 隧道 → `secrets.json` 里的固定域名（模型名 `glm-5.3-flash`）
6. 对话测试
7. 看日志 / 用完关机
8. **循环日志 + 监控**：一直运行，每分钟打印新完成的请求，每 5 分钟自检一次；进程退出时自动重启

实测（G4）：单人生成约 45 tok/s；读长输入约 560 tok/s（10.4 万 token 用 187 秒，大海捞针通过）；显存约 77–79GB。

**限制和已知问题**
- **一次只处理一个请求**，多人同时用要排队。
- 长请求要用**流式**：非流式在约 140 秒时会被 Cloudflare 断开。
- 思考很长（常有几千字，英文），简单问题也要先等 1–2 分钟才出正文。

**这里绕过的坑**（已写在代码里，一般不用管）
- Blackwell 必须用 `CMAKE_CUDA_ARCHITECTURES=120a` 编译，用 `120` 会在 mxfp4 kernel 上报错。
- 新版 Strata 不认 GLM 的分词规则（`glm4`），第 3 步会给 `serve_glm.py` 打一个小补丁。
- 必须用 `--slots 9072` 固定常驻显存的专家数：第二次启动时引擎会按使用统计多放专家，第一个请求就爆显存（`c_stage: out of memory`）。
- 不用原仓库的 `glm-engine.sh`：它依赖 `flock` 和 `systemd-run`，Colab 里没有。
- Maya-M（116GB）跑不了：它的专家量化格式组合（IQ2_S + IQ3_S）Strata-GLM 没有对应 kernel。


In [ ]:
# 1. 挂载 Drive，从 Hugging Face 下载模型 + 去审查 LoRA
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
MODEL_DIR = "/content/models/glm"
MODEL_PATH = f"{MODEL_DIR}/Maya-S-v2-IQ2_XXS/GLM-5.3-Flash-Maya-S-v2-IQ2_XXS-00001-of-00003.gguf"
LORA_PATH = f"{MODEL_DIR}/lora/GLM-5.3-Flash-Ablitered2-LoRA-v2.gguf"
MODEL_ALIAS = "glm-5.3-flash"
BIN_DIR = "/content/drive/MyDrive/models/bin"

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U "huggingface_hub[hf_xet]"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"      # 多线程高速下载
!hf download peasantsmith/GLM-5.3-Flash-Maya-GGUF --include "Maya-S-v2-IQ2_XXS/*" --local-dir {MODEL_DIR}
!hf download GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF GLM-5.3-Flash-Ablitered2-LoRA-v2.gguf --local-dir {MODEL_DIR}/lora
!du -sh {MODEL_DIR}/*


In [ ]:
# 2. 准备 Strata-GLM 引擎。代码固定在测试过的版本；编译结果缓存在 Drive（约 670MB）
import os, subprocess
G = "/content/Strata-GLM"
STRATA_GLM_COMMIT = "94ac0227dc3e75635eccbe2c6e44476fcdcfb8c5"
STRATA_COMMIT = "37d5c9e84514150ee74d69f4f0f0466ba4b91b51"      # serve_glm.py 依赖的 Strata 服务端
LLAMA_COMMIT = "3cf03257f219afbe7334045ff7c6a06ac68c627d"       # Strata-GLM 指定的 ggml 版本
CACHED = f"{BIN_DIR}/glm_generate-sm120a"
ENGINE = f"{G}/build-glm-gpu/glm_generate"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode: raise RuntimeError(f"失败：{cmd}")

!pip install -q ninja regex numpy jinja2 pyyaml gguf
if not os.path.isdir(G):
    sh(f"git clone -q https://github.com/Indras-Mirror/Strata-GLM {G} && git -C {G} checkout -q {STRATA_GLM_COMMIT}")
if not os.path.isdir("/root/AI/Strata"):
    sh(f"mkdir -p /root/AI && git clone -q https://github.com/Niko1221/Strata /root/AI/Strata && git -C /root/AI/Strata checkout -q {STRATA_COMMIT}")

if os.path.isfile(CACHED) and os.path.getsize(CACHED) > 100_000_000:
    print("复用 Drive 里的编译结果:", CACHED)
    sh(f"mkdir -p {G}/build-glm-gpu && cp '{CACHED}' {ENGINE} && chmod +x {ENGINE}")
else:
    print("Drive 里没有编译结果，开始编译（约 25 分钟）...")
    if not os.path.isdir(f"{G}/third_party/llama.cpp"):
        sh(f"git clone -q https://github.com/ggml-org/llama.cpp {G}/third_party/llama.cpp && git -C {G}/third_party/llama.cpp checkout -q {LLAMA_COMMIT}")
        sh(f"patch -d {G}/third_party/llama.cpp -p1 < {G}/tools/glm/patches/0001-ggml-cuda-no-graph-for-batched-passes.patch")
    # Blackwell 必须用 120a（arch-specific），120 会在 mxfp4 MMQ 上报 "mma with block scale not supported"
    sh(f"cmake -S {G} -B {G}/build-glm-gpu -G Ninja -DCMAKE_BUILD_TYPE=Release -DSTRATA_ENABLE_CUDA=ON -DSTRATA_GGML_CUDA=ON "
       f"-DSTRATA_MMQ_KQUANTS=ON -DCMAKE_CUDA_ARCHITECTURES=120a -DSTRATA_GGML_DIR={G}/third_party/llama.cpp > /content/glm_cmake.log 2>&1")
    sh(f"ninja -C {G}/build-glm-gpu -j $(nproc) glm_generate > /content/glm_build.log 2>&1 || (tail -30 /content/glm_build.log; false)")
    sh(f"mkdir -p '{BIN_DIR}' && cp {ENGINE} '{CACHED}.tmp' && mv '{CACHED}.tmp' '{CACHED}'")
    print("✅ 编译完成并已存到 Drive:", CACHED)
!ls -la {ENGINE}


In [ ]:
# 3. 启动服务（端口 8001，带 API key，只监听本机；对外由第 4 步的保活代理在 8000 端口转发）
import os, time, subprocess, urllib.request, secrets, json

def get_secret(name):
    """先读 Colab Secrets（左侧钥匙图标），再读 Drive 里的 MyDrive/models/secrets.json"""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    try:
        return json.load(open("/content/drive/MyDrive/models/secrets.json")).get(name)
    except Exception:
        return None

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"找不到 {MODEL_PATH}：请先运行第 1 步")
API_KEY = get_secret("LLM_API_KEY") or secrets.token_urlsafe(24)
print("API key:", "固定（来自 secrets）" if get_secret("LLM_API_KEY") else "本次随机生成")

TOK_DIR = f"{MODEL_DIR}/tok"
if not os.path.exists(f"{TOK_DIR}/vocab.json"):     # 从 GGUF 导出 tokenizer + 聊天模板
    !python3 {G}/tools/glm/serve/export_tokenizer.py {MODEL_PATH} {TOK_DIR}

# 补丁：新版 Strata 不认 'glm4' 分词规则（直接拒绝加载），且改为用 self._split 切词。
# 先按 qwen35 建好，再换成 GLM4 的切分正则（Strata-GLM 原本就用这个正则，只是接口变了）
p = f"{G}/tools/glm/serve/serve_glm.py"; s = open(p).read()
old = ("        super().__init__(tokens, merges, types, pre=\"glm4\", special_ids=special_ids)\n"
       "        self._re = glm_tok.regex.compile(glm_tok.GLM4_PATTERN)\n")
new = ("        super().__init__(tokens, merges, types, pre=\"qwen35\", special_ids=special_ids)\n"
       "        self.pre = \"glm4\"\n"
       "        self._re = glm_tok.regex.compile(glm_tok.GLM4_PATTERN)\n"
       "        self._split = self._re.findall\n")
if old in s:
    open(p, "w").write(s.replace(old, new)); print("已给 serve_glm.py 打分词补丁")

CTX = 524288            # 上下文：512K（作者实测配置）
SLOTS = 9072            # 常驻显存的专家数。不固定的话，第二次启动会按使用统计多放专家，第一个请求就爆显存
ARENA_GIB = 100         # 内存池上限：放显存放不下的专家
cfg = {"exe": ENGINE,
       "args": ["-m", MODEL_PATH, "--backend", "cuda", "--experts", "gpu", "--slots", str(SLOTS),
                "--arena-gib", str(ARENA_GIB), "--arena-skip-resident", "--ctx", str(CTX),
                "--prefill-chunk", "4096", "--chunk-mmq", "--vram-margin", "7.5", "--vram-grow", "1.5",
                "--prune", f"{G}/bench/glm-2026-10-09/prune-ezct-0.25.txt", "--prune-penalty", "0.05", "--arena-adapt",
                "--skip-miss", "0.15", "--skip-file", "0.15", "--skip-file-prefill", "0.15", "--renorm-skip",
                "--pf-b", "1.0", "--pcie", "auto",
                "--census", "/content/glm-maya-s.census",          # 专家使用统计，越用越准
                "--stop", "154820,154827,154829",
                "--lora", LORA_PATH, "--lora-exps"],               # 去审查 LoRA；去掉这两项就是原版
       "cwd": G, "tokenizer": TOK_DIR, "model_name": MODEL_ALIAS, "log": "/content/glm-engine.log",
       "port": 8001, "gpu": 0, "sampling": {"temperature": 1.0, "top_p": 0.95}}
json.dump(cfg, open("/content/glm.json", "w"), indent=1)

SERVE_CMD = ["python3", f"{G}/tools/glm/serve/serve_glm.py", "--engine", "strata", "--config", "/content/glm.json",
             "--tokenizer", TOK_DIR, "--port", "8001", "--api-key", API_KEY]

def api_ok(url, key=None, timeout=10):
    try:
        h = {"User-Agent": "colab-llm-monitor/1.0"}   # Cloudflare 会拦 Python 默认 UA（403 / error 1010）
        if key: h["Authorization"] = f"Bearer {key}"
        return urllib.request.urlopen(urllib.request.Request(url, headers=h), timeout=timeout).status == 200
    except Exception:
        return False

def start_server():
    subprocess.run(["pkill", "-f", "serve_glm.py"]); subprocess.run(["pkill", "-f", "glm_generate"])
    for _ in range(60):          # 等旧引擎完全退出、释放显存，否则新进程会显存不够
        if subprocess.run(["pgrep", "-f", "glm_generate"], capture_output=True).returncode != 0: break
        time.sleep(1)
    p = subprocess.Popen(SERVE_CMD, stdout=open("/content/glm-serve.log", "a"), stderr=subprocess.STDOUT,
                         start_new_session=True)
    for i in range(180):
        if api_ok("http://127.0.0.1:8001/v1/models", API_KEY, 5):
            print(f"服务已就绪（{i*5}s）"); return p
        if p.poll() is not None:
            print("启动失败，日志："); break
        time.sleep(5)
    return p

proc = start_server()
# 服务就绪后引擎还在加载内存池；第一个请求会等它加载完（首次约 5 分钟）
!tail -n 5 /content/glm-serve.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


In [ ]:
# 4. 保活代理：8000 → GLM 服务 8001
#     超长上下文时 引擎读长输入 可能超过 100 秒，Cloudflare 会因"100 秒无响应"断开（524）。
#     代理在上游 20 秒没响应时先回 200，然后每 15 秒发一次保活（流式发 SSE 注释行，非流式发前导空格），
#     客户端都会自动忽略。隧道、第 4 步、Mac/Magpie 的地址都不用改。
#     注意：已提前回 200 之后，如果服务报错，状态码仍是 200，错误放在正文里。
import importlib.util
if importlib.util.find_spec("aiohttp") is None:
    !pip install -q aiohttp

PROXY_CODE = r'''"""保活反向代理：挡在 llama-server 前面，避免 Cloudflare 100 秒无响应断开（524）。
上游超过 GRACE 秒没回响应头时，先回 200，然后每 INTERVAL 秒发保活：
  - 流式（SSE）：发 ": keepalive" 注释行，客户端会忽略
  - 非流式（JSON）：在正文前发空格，JSON 解析器会忽略
"""
import asyncio, json, os, sys
from aiohttp import web, ClientSession, ClientTimeout, TCPConnector

LISTEN = int(os.environ.get("PROXY_PORT", "8000"))
UPSTREAM = os.environ.get("UPSTREAM", "http://127.0.0.1:8001")
GRACE = float(os.environ.get("KEEPALIVE_GRACE", "20"))
INTERVAL = float(os.environ.get("KEEPALIVE_INTERVAL", "15"))

HOP = {"host", "content-length", "transfer-encoding", "connection", "keep-alive",
       "accept-encoding", "content-encoding", "upgrade", "te", "trailer", "proxy-connection"}


def log(*a):
    print(*a, file=sys.stderr, flush=True)


async def pump(session, request, body, q):
    """读上游响应，按顺序放进队列：head → data... → end / error"""
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
    try:
        async with session.request(request.method, UPSTREAM + request.path_qs,
                                   headers=headers, data=body) as resp:
            await q.put(("head", resp.status, resp.headers))
            async for chunk in resp.content.iter_any():
                await q.put(("data", chunk))
        await q.put(("end",))
    except asyncio.CancelledError:
        raise
    except Exception as e:
        await q.put(("error", f"{type(e).__name__}: {e}"))


async def handler(request):
    body = await request.read()
    wants_sse = False
    if body:
        try:
            wants_sse = bool(json.loads(body).get("stream"))
        except Exception:
            pass

    q = asyncio.Queue()
    task = asyncio.create_task(pump(request.app["session"], request, body, q))
    out = None             # 已发给客户端的响应
    sse = wants_sse
    wrote_body = False     # 是否已经转发过上游正文
    at_boundary = True     # SSE：上次转发的内容是否以空行结尾（只在事件之间插保活）

    async def commit(status, ctype, extra=None):
        nonlocal out
        h = {"Content-Type": ctype, "Cache-Control": "no-cache, no-transform",
             "X-Accel-Buffering": "no"}
        h.update(extra or {})
        out = web.StreamResponse(status=status, headers=h)
        await out.prepare(request)

    try:
        while True:
            try:
                item = await asyncio.wait_for(q.get(), GRACE if out is None else INTERVAL)
            except asyncio.TimeoutError:
                if out is None:   # 上游迟迟没回响应头：先回 200，开始保活
                    await commit(200, "text/event-stream" if sse else "application/json")
                    log(f"[keepalive] {request.method} {request.path} 上游 {GRACE:.0f}s 未响应，提前回 200")
                if sse and at_boundary:
                    await out.write(b": keepalive\n\n")
                elif not sse and not wrote_body:
                    await out.write(b" ")
                continue

            kind = item[0]
            if kind == "head":
                _, status, h = item
                ctype = h.get("Content-Type", "application/octet-stream")
                if out is None:
                    sse = "event-stream" in ctype
                    extra = {k: v for k, v in h.items()
                             if k.lower() not in HOP and k.lower() not in
                             ("content-type", "cache-control")}
                    await commit(status, ctype, extra)
                elif status != 200:
                    log(f"[keepalive] 上游返回 {status}，但已提前回了 200，错误内容会放在正文里")
            elif kind == "data":
                chunk = item[1]
                await out.write(chunk)
                wrote_body = True
                at_boundary = chunk.endswith(b"\n\n") or chunk.endswith(b"\r\n\r\n")
            elif kind == "end":
                await out.write_eof()
                return out
            else:  # error
                msg = item[1]
                log(f"[keepalive] 上游出错: {msg}")
                err = {"error": {"message": f"proxy upstream error: {msg}", "type": "proxy_error"}}
                if out is None:
                    return web.json_response(err, status=502)
                payload = json.dumps(err).encode()
                await out.write((b"data: " + payload + b"\n\n") if sse else
                                (payload if not wrote_body else b""))
                await out.write_eof()
                return out
    except ConnectionResetError:
        log(f"[keepalive] 客户端断开 {request.method} {request.path}")
        return out
    except asyncio.CancelledError:
        log(f"[keepalive] 客户端断开 {request.method} {request.path}")
        raise
    finally:
        task.cancel()   # 客户端断开时也断开上游，llama-server 会停止这次生成


async def on_startup(app):
    app["session"] = ClientSession(timeout=ClientTimeout(total=None, sock_read=None),
                                   connector=TCPConnector(limit=0), auto_decompress=False)


async def on_cleanup(app):
    await app["session"].close()


app = web.Application(client_max_size=1024 ** 3)   # 长上下文请求体可能很大
app.on_startup.append(on_startup)
app.on_cleanup.append(on_cleanup)
app.router.add_route("*", "/{tail:.*}", handler)

if __name__ == "__main__":
    log(f"keepalive proxy :{LISTEN} -> {UPSTREAM} (grace {GRACE}s, interval {INTERVAL}s)")
    web.run_app(app, host="127.0.0.1", port=LISTEN, access_log=None, print=None)
'''
open("/content/keepalive_proxy.py", "w").write(PROXY_CODE)

subprocess.run(["pkill", "-f", "keepalive_proxy.py"]); time.sleep(1)   # 停掉旧代理
plog = open("/content/keepalive_proxy.log", "w")
proxy_proc = subprocess.Popen(["python3", "/content/keepalive_proxy.py"],
                              env={**os.environ, "PROXY_PORT": "8000", "UPSTREAM": "http://127.0.0.1:8001"},
                              stdout=plog, stderr=subprocess.STDOUT, start_new_session=True)
ok = False
for _ in range(30):
    try:
        ok = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).status == 200
        if ok: break
    except Exception:
        pass
    if proxy_proc.poll() is not None: break
    time.sleep(1)
if ok:
    print("✅ 保活代理已就绪：127.0.0.1:8000 → GLM :8001")
else:
    print("❌ 代理没起来（或 GLM 服务没在 8001 运行），日志：")
    !tail -n 20 /content/keepalive_proxy.log


In [ ]:
# 5. 建立 Cloudflare 隧道 （读取 Drive 里的 models/secrets.json，用 scripts/setup_tunnel.sh 生成）
#    没配 secrets 时自动退回临时 trycloudflare 地址
import re
subprocess.run(["pkill", "-f", "cloudflared"])   # 停掉旧隧道（用 Python 调，避免 pkill 误杀自己所在的 shell）
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tlog = open("/content/cloudflared.log", "w")
CF_TOKEN, CF_HOST = get_secret("CF_TUNNEL_TOKEN"), get_secret("CF_HOSTNAME")

if CF_TOKEN and CF_HOST:
    TUNNEL_CMD = ["/content/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                  "--url", "http://127.0.0.1:8000", "run", "--token", CF_TOKEN]
    tunnel = subprocess.Popen(TUNNEL_CMD, stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = f"https://{CF_HOST}"
    for _ in range(30):
        if "Registered tunnel connection" in open("/content/cloudflared.log").read(): break
        time.sleep(1)
    else:
        print("固定隧道可能没连上，日志："); PUBLIC_URL = None
        !tail -n 20 /content/cloudflared.log
else:
    print("⚠️ 没读到 secrets.json，使用临时地址")
    TUNNEL_CMD = ["/content/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                  "--url", "http://127.0.0.1:8000"]
    tunnel = subprocess.Popen(TUNNEL_CMD, stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m: PUBLIC_URL = m.group(0); break
        time.sleep(1)
    if not PUBLIC_URL:
        print("隧道没起来，日志：")
        !tail -n 20 /content/cloudflared.log

if PUBLIC_URL:
    fixed = bool(CF_TOKEN and CF_HOST and get_secret("LLM_API_KEY"))
    print("✅ 固定地址 + 固定 key，本地 .env 不用改\n" if fixed
          else "把下面三行覆盖写入 Mac 上的 ~/探索/colab/.env ：\n")
    print(f"COLAB_LLM_URL={PUBLIC_URL}/v1")
    if not fixed: print(f"COLAB_LLM_KEY={API_KEY}")
    print(f"COLAB_LLM_MODEL={MODEL_ALIAS}")
    # 自检：从公网地址访问一次
    time.sleep(5)
    try:
        r = urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + "/v1/models",
                headers={"Authorization": "Bearer " + API_KEY,
                         "User-Agent": "colab-llm/1.0"}), timeout=20)
        print("公网自检:", r.status, json.loads(r.read())["data"][0]["id"])
    except Exception as e:
        print("公网自检失败（可再等几十秒重试第 5 步）:", e)

In [ ]:
# 6. 对话测试（流式）。默认只打印正文；show_thinking=True 时也打印思考（英文）
!pip install -q openai
import time
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key=API_KEY, timeout=1800)

def chat(prompt, max_tokens=4000, show_thinking=False):
    t0, first, n, thinking = time.time(), None, 0, 0
    stream = client.chat.completions.create(
        model=MODEL_ALIAS, stream=True, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}], stream_options={"include_usage": True})
    for chunk in stream:
        if chunk.usage: n = chunk.usage.completion_tokens
        if not chunk.choices: continue
        d = chunk.choices[0].delta
        r = getattr(d, "reasoning_content", None) or ""
        c = d.content or ""
        if (r or c) and first is None: first = time.time()
        thinking += len(r)
        if show_thinking and r: print(r, end="", flush=True)
        if c: print(c, end="", flush=True)
    dt = time.time() - (first or t0)
    print(f"\n\n--- 首字 {(first or t0)-t0:.1f}s | {n} tokens（其中思考约 {thinking} 字）| {n/max(dt,1e-6):.1f} tok/s ---")

chat("用三句话介绍一下你自己。")


In [ ]:
# 7.（可选）看日志
!tail -n 20 /content/glm-loop.log
!tail -n 20 /content/glm-serve.log
!tail -n 15 /content/glm-engine.log
!tail -n 10 /content/keepalive_proxy.log
!tail -n 10 /content/cloudflared.log

# 用完关机：先停掉第 8 步的循环日志，再取消下面两行注释并运行，立刻释放 GPU，停止消耗计算单元
# from google.colab import runtime
# runtime.unassign()


In [ ]:
# 8. 循环日志 + 监控：最后运行，让它一直跑着（cell 一直在执行，Colab 就不会因为「空闲」回收运行时）。
#    每 LOG_EVERY 秒打印这段时间新完成的请求（耗时、token 数、速度）、出错和正在进行的请求；
#    每 CHECK_EVERY 秒自检一次服务 / 代理 / 隧道 / 公网，进程退出或连续 3 次无响应时自动重启。
#    想停时点 cell 左边的停止按钮（只停这个循环，服务和隧道继续运行）。
#    日志同时写到 /content/glm-loop.log；第 7 步也会显示最后几行。
import time, re, subprocess, os

LOG_EVERY = 60
CHECK_EVERY = 300
SERVE_LOG = "/content/glm-serve.log"
KEEP = re.compile(r"done:|engine stopped|engine had stopped|engine is running|starting the engine|"
                  r"error|Error|out of memory|Traceback")
PROGRESS = ("reading the prompt", "thinking:", "answering:")

def new_lines(path, pos):
    """读 path 里 pos 之后新写入的完整行，返回 (行列表, 新位置)"""
    try:
        if os.path.getsize(path) < pos: pos = 0            # 日志被清空过
        with open(path, "rb") as f:
            f.seek(pos); data = f.read()
        end = data.rfind(b"\n") + 1                        # 只取完整的行，半行留到下次
        return data[:end].decode("utf-8", "replace").splitlines(), pos + end
    except FileNotFoundError:
        return [], pos

LOOP_LOG = "/content/glm-loop.log"            # 同时写一份到文件：网页断开后也能看（!tail /content/glm-loop.log）

def stamp(): return time.strftime("%m-%d %H:%M:%S")
def clean(l): return l.replace("[strata] ", "").strip()
def out(msg):
    line = f"{stamp()}  {msg}"
    print(line, flush=True)
    with open(LOOP_LOG, "a") as f: f.write(line + "\n")

log_pos = os.path.getsize(SERVE_LOG) if os.path.exists(SERVE_LOG) else 0
bad, last_check, n_req = 0, 0.0, 0
out(f"循环日志开始：每 {LOG_EVERY}s 打印新请求，每 {CHECK_EVERY}s 自检")
while True:
    # ---- 这段时间的请求：完成 / 出错照打；还没完成的只打最新一条进度
    lines, log_pos = new_lines(SERVE_LOG, log_pos)
    last_prog = None
    for l in lines:
        if KEEP.search(l):
            if "done:" in l: n_req += 1
            out(clean(l)[:200])
            last_prog = None
        elif any(k in l for k in PROGRESS):
            last_prog = l
    if last_prog:
        out(f"进行中: {clean(last_prog)[:160]}")

    # ---- 定时自检
    if time.time() - last_check >= CHECK_EVERY:
        last_check = time.time(); msgs = []
        ok_srv = proc.poll() is None and api_ok("http://127.0.0.1:8001/v1/models", API_KEY, 30)
        bad = 0 if ok_srv else bad + 1
        if proc.poll() is not None or bad >= 3:       # 读超长输入时可能短暂无响应，连续 3 次才重启
            msgs.append("GLM 服务异常，重启"); proc = start_server(); bad = 0
        elif bad:
            msgs.append(f"GLM 服务无响应（第 {bad} 次，可能在读长输入）")
        if proxy_proc.poll() is not None:
            msgs.append("保活代理退出，重启")
            proxy_proc = subprocess.Popen(["python3", "/content/keepalive_proxy.py"],
                                          env={**os.environ, "PROXY_PORT": "8000", "UPSTREAM": "http://127.0.0.1:8001"},
                                          stdout=open("/content/keepalive_proxy.log", "a"), stderr=subprocess.STDOUT,
                                          start_new_session=True)
        if tunnel.poll() is not None:
            msgs.append("隧道退出，重启")
            tunnel = subprocess.Popen(TUNNEL_CMD, stdout=open("/content/cloudflared.log", "a"), stderr=subprocess.STDOUT,
                                      start_new_session=True)
        public = api_ok(PUBLIC_URL + "/v1/models", API_KEY, 30) if PUBLIC_URL else None
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
        status = "；".join(msgs) if msgs else "服务 + 代理 + 隧道正常"
        out(f"[自检] {status} | 公网 {'OK' if public else ('失败' if public is False else '-')}"
            f" | 显存 {used} | 累计完成 {n_req} 个请求")
    time.sleep(LOG_EVERY)
